# Lab 11 · the LightGBM re-ranker

The runnable companion of the lab page, `docs/labs/11-lgbm-rerank.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "lgbm_rerank"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
from recbench.methods.rerank import FEATURES

print(model.fit_info)  # training users and how often their chosen item was among the candidates
sorted(zip(model.model.feature_importances_, FEATURES), reverse=True)[:10]

**Exercise.** What share of the validation users' chosen items were among their candidates at all (the ceiling)?

<details><summary>Hint</summary>

`lab.once(METHOD, DATASET, best=True)` returns the stored metrics; look for `candidate_recall`.

</details>

<details><summary>Solution</summary>

```python
lab.once(METHOD, DATASET, best=True)["metrics"]["candidate_recall"]
```
No ranker can do better than this share: improving the candidates (Level 4) raises it.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "rerank_candidates", [100, 200], start="best")
lab.plot_sweep(table)
table

**Your prediction for `rerank_candidates`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "lgbm_leaves", [15, 31, 63, 127], start="best")
lab.plot_sweep(table)
table

**Your prediction for `lgbm_leaves`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh lgbm_rerank:more-train-users
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `more-train-users` is ready to run; `downsample`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_lgbm_rerank_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:more-train-users")

In [ ]:
lab.compare(METHOD, f"{METHOD}:downsample")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`third-generator` in `labs/11-lgbm-rerank/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:third-generator")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:third-generator", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.